# Anki csv creation for Databases

This notebook doesn't belong in this repo but it shall here for a while until
its own repo is created.

In [1]:
import pandas as pd

In [2]:
excel_file = pd.ExcelFile("../../assets/Databases.xlsx")
with excel_file as xls:
    df = pd.read_excel(xls, "Databases")

In [3]:
df = df.head(3)
df

,Name,Vendor,Country of Origin,Start Year,Source Type,License,Query Language,Source Language,Storage Engines,Generation,...,Query Execution,Parallelism,CAP Inclination,User-Defined Functions,System Architecture,MapReduce,Partitioning Scheme,Partitioning Methods,Notable Features,Typical Use Case
0,MySQL,Oracle,Sweden,1994.0,OS,GPL-2.0,SQL,C\nC++,InnoDB [default]\nMyISAM\n[others],SQL,...,tuple-at-a-time,Intra-Query (limited)\nInter-Query\nIntra-Oper...,CA,No,Shared-Everything,No,NaN,horizontal partitioning\nsharding (MySQL Clust...,NaN,Consistent data. Ex: Financial
1,PostgreSQL,(none),USA,1986.0,OS,PostgreSQL License,SQL,C,NaN,SQL,...,tuple-at-a-time,Intra-Query\nInter-Query\nIntra-Operator,CA,PL/SQL\nPython\nPerl\nTCL,Shared-Everything,No,NaN,range\nlist\nhash,NaN,Consistent data. Ex: Financial
2,MongoDB,MongoDB Inc,USA,2009.0,"Commercial, OS",Server Side Public License,MQL,C++\nJavascript\nPython,WiredTiger [default]\nInMemory,NoSQL,...,tuple-at-a-time,Intra-Query\nInter-Query\nIntra-Operator,CP,No,Shared-Nothing,Yes,Sharding,range\nzone\nhash\ncomposite,NaN,NaN


In [4]:
corrected = df.copy()
corrected["Start Year"] = corrected["Start Year"].astype(int)
corrected

,Name,Vendor,Country of Origin,Start Year,Source Type,License,Query Language,Source Language,Storage Engines,Generation,...,Query Execution,Parallelism,CAP Inclination,User-Defined Functions,System Architecture,MapReduce,Partitioning Scheme,Partitioning Methods,Notable Features,Typical Use Case
0,MySQL,Oracle,Sweden,1994,OS,GPL-2.0,SQL,C\nC++,InnoDB [default]\nMyISAM\n[others],SQL,...,tuple-at-a-time,Intra-Query (limited)\nInter-Query\nIntra-Oper...,CA,No,Shared-Everything,No,NaN,horizontal partitioning\nsharding (MySQL Clust...,NaN,Consistent data. Ex: Financial
1,PostgreSQL,(none),USA,1986,OS,PostgreSQL License,SQL,C,NaN,SQL,...,tuple-at-a-time,Intra-Query\nInter-Query\nIntra-Operator,CA,PL/SQL\nPython\nPerl\nTCL,Shared-Everything,No,NaN,range\nlist\nhash,NaN,Consistent data. Ex: Financial
2,MongoDB,MongoDB Inc,USA,2009,"Commercial, OS",Server Side Public License,MQL,C++\nJavascript\nPython,WiredTiger [default]\nInMemory,NoSQL,...,tuple-at-a-time,Intra-Query\nInter-Query\nIntra-Operator,CP,No,Shared-Nothing,Yes,Sharding,range\nzone\nhash\ncomposite,NaN,NaN


### Standard melts

In [5]:
SKIPPED_COLUMNS = ["Name", "Notable Features"]

melted = corrected.melt(
    id_vars=["Name"],
    value_vars=list(
        filter(lambda n: n not in SKIPPED_COLUMNS, df.columns.to_list())
    ),
).dropna()
melted.insert(0, "key", melted["Name"] + ":\n" + melted["variable"])
melted = melted.drop(columns=["Name", "variable"])
melted

,key,value
0,MySQL:\nVendor,Oracle
1,PostgreSQL:\nVendor,(none)
2,MongoDB:\nVendor,MongoDB Inc
3,MySQL:\nCountry of Origin,Sweden
4,PostgreSQL:\nCountry of Origin,USA
...,...,...
111,MySQL:\nPartitioning Methods,horizontal partitioning\nsharding (MySQL Clust...
112,PostgreSQL:\nPartitioning Methods,range\nlist\nhash
113,MongoDB:\nPartitioning Methods,range\nzone\nhash\ncomposite
114,MySQL:\nTypical Use Case,Consistent data. Ex: Financial


### Grouped values

This one groups database names according to common values. For instance, it adds a row for all databases that have the data model "Relational"

In [11]:
initial = corrected.copy()
FILTERED = ["Name"]
grouped = pd.DataFrame()

column_list = list(filter(lambda n: n not in FILTERED, initial.columns))
for grouper in column_list:
    if initial[grouper].dtype == "object":
        current = initial[["Name", grouper]]
        current = current.assign(
            **{grouper: current[grouper].str.split("\n")}
        ).dropna()
        current["count"] = current[grouper].apply(lambda a: len(a))
        current = current[current["count"] > 1]
        current = current[["Name", grouper]].explode(grouper).dropna()
        current = current[current[grouper] != "[others]"]
        current = (
            current.groupby(grouper)["Name"]
            .apply(lambda v: "\n".join(v))
            .reset_index()
        )

        current[grouper] = (
            grouper
            + " includes:\n"
            + current[grouper].replace(r"\[.*\]", "", regex=True)
        )
        current = current.rename(columns={grouper: "key", "Name": "value"})
        current = current[["key", "value"]]
        grouped = pd.concat([current, grouped], ignore_index=True)
grouped

,key,value
0,Partitioning Methods includes:\ncomposite,MongoDB
1,Partitioning Methods includes:\nhash,PostgreSQL\nMongoDB
2,Partitioning Methods includes:\nhorizontal par...,MySQL
3,Partitioning Methods includes:\nlist,PostgreSQL
4,Partitioning Methods includes:\nrange,PostgreSQL\nMongoDB
...,...,...
69,Storage Engines includes:\nWiredTiger,MongoDB
70,Source Language includes:\nC,MySQL
71,Source Language includes:\nC++,MySQL\nMongoDB
72,Source Language includes:\nJavascript,MongoDB


### Merge all

In [9]:
merged = pd.concat([melted, grouped])

### Export

In [10]:
merged.to_csv(
    "../../artifacts/databases-new.csv",
    ":",
    encoding="UTF-8",
    header=False,
    index=False,
)